
$$
\min \quad
\frac{w_1}{Risk_{\max}} \sum_{l \in L} \sum_{e \in E} Risk_{e,\, Class_l} \cdot f_{l,e}
\;+\;
\frac{w_2}{Cost_{\max}} \left(
  \sum_{v \in V} FC_v \cdot z_v
  \;+\;
  \sum_{l \in L} \sum_{v \in V} \sum_{e \in E} VC_{v,e} \cdot y_{l,v} \cdot \frac{1}{|V|}
\right)
$$

In [ ]:
import pulp as pl
import pandas as pd
import numpy as np
import read_csv
from shapely import wkt
import geopandas as gpd
from shapely.geometry import LineString


In [2]:
%pip install geopandas, LineString, wkt

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: 'geopandas,'


In [ ]:
%pip install gc, pickle


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement gc (from versions: none)
ERROR: No matching distribution found for gc


Daten

In [2]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
import gc 

# --- 1. Chunked Loading & Filtern der Kanten ---
edges_file = r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\data\data_jonas\edges_berlin.csv"
nodes_file = r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\data\data_jonas\nodes_berlin.csv"

allowed_highways = [
    'motorway', 'motorway_link', 
    'trunk', 'trunk_link', 
    'primary', 'primary_link', 
    'secondary', 'secondary_link'
]

print("Lese und filtere edges.csv in Chunks")

# Immer 5 Millionen Zeilen auf einmal 
chunk_size = 5_000_000
filtered_chunks = []

# mit usecols laden wir nur die Spalten die wir brauchen 
for chunk in pd.read_csv(edges_file, chunksize=chunk_size, usecols=['u', 'v', 'highway']):
    chunk_filtered = chunk[chunk['highway'].isin(allowed_highways)]
    filtered_chunks.append(chunk_filtered)

# Alle Chunks zusammenfügen und RAM freigeben 
edges_df = pd.concat(filtered_chunks, ignore_index=True)
del filtered_chunks 
gc.collect()

print(f" {len(edges_df)} Kanten nach dem Filtern übrig.")


#Filtern und laden der Knoten, die in den bereits gefilterten Kanten vorkommen 
print("Lese und filtere nodes")
relevant_nodes = set(edges_df['u']).union(set(edges_df['v']))

filtered_node_chunks = []
for chunk in pd.read_csv(nodes_file, chunksize=chunk_size, index_col="node_id"):
    # Behalte nur Knoten, deren ID in relevant_nodes ist
    chunk_filtered = chunk[chunk.index.isin(relevant_nodes)]
    filtered_node_chunks.append(chunk_filtered)

#Wieder alles zusammenfügen und RAM freigeben
nodes_df = pd.concat(filtered_node_chunks)
del filtered_node_chunks
del relevant_nodes
gc.collect()

print(f"{len(nodes_df)} Knoten übrig.")


# Koordinaten mergen
print("Füge Koordinaten an Kanten an")
edges_df = edges_df.merge(nodes_df, left_on='u', right_index=True)
edges_df = edges_df.rename(columns={'lat': 'lat_u', 'lon': 'lon_u'})

edges_df = edges_df.merge(nodes_df, left_on='v', right_index=True)
edges_df = edges_df.rename(columns={'lat': 'lat_v', 'lon': 'lon_v'})


# Geometrien erstellen
print("Erstelle Geometrien")
geometries = [
    LineString([(lon_u, lat_u), (lon_v, lat_v)]) 
    for lon_u, lat_u, lon_v, lat_v 
    in zip(edges_df['lon_u'], edges_df['lat_u'], edges_df['lon_v'], edges_df['lat_v'])
]

edges_gdf = gpd.GeoDataFrame(edges_df, geometry=geometries, crs="EPSG:4326")
edges_gdf = edges_gdf.drop(columns=['lat_u', 'lon_u', 'lat_v', 'lon_v'])

#Aufräumen, damit nicht alles im RAM liegt 
del edges_df
del geometries
gc.collect()


# Projektion & OR-Sets 
print("Projiziere und erstelle Dictionaries")
edges_gdf = edges_gdf.to_crs("EPSG:32632")
edges_gdf['Dist_km'] = edges_gdf.geometry.length / 1000

E = list(zip(edges_gdf['u'], edges_gdf['v'])) 
N = list(set(edges_gdf['u']).union(set(edges_gdf['v']))) 

edges_gdf = edges_gdf.set_index(['u', 'v']) 
Dist = edges_gdf['Dist_km'].to_dict()

print(f"Erfolgreich geladen: {len(E)} Kanten und {len(N)} Knoten.")



Lese und filtere edges.csv in Chunks
 94391 Kanten nach dem Filtern übrig.
Lese und filtere nodes
71817 Knoten übrig.
Füge Koordinaten an Kanten an
Erstelle Geometrien
Projiziere und erstelle Dictionaries
Erfolgreich geladen: 94391 Kanten und 71817 Knoten.


In [13]:
import pickle
import os

pickle_path = os.path.join("Sets/", "pulp_data_berlin.pkl")

pulp_data = {
    'E': E,
    'N': N
}

with open(pickle_path, 'wb') as f:
    pickle.dump(pulp_data, f)

In [14]:
with open(pickle_path, 'rb') as f:
    pulp_data = pickle.load(f)

print(f"Gespeicherte Daten erfolgreich geladen: {len(pulp_data['E'])} Kanten und {len(pulp_data['N'])} Knoten.")

Gespeicherte Daten erfolgreich geladen: 94391 Kanten und 71817 Knoten.


Risikodaten

In [17]:
%pip install read_csv

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement read_csv (from versions: none)
ERROR: No matching distribution found for read_csv


In [1]:
# 1. Kanten aus dem Straßennetz als Set
network_edges = set(edges_gdf.index)

# 2. Kanten aus der Unfall-Datei als Set
# Wir prüfen beide Richtungen (u->v und v->u), um Richtungsprobleme zu vermeiden!
# acc_df = read_csv(r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\data\processed\Accidents.csv",  sep=";", decimal=",")

acc_df = r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\data\processed\Accidents.csv"

acc_gdf = gpd.GeoDataFrame(
    acc_df,
    geometry=gpd.points_from_xy(acc_df.XGCSWGS84, acc_df.YGCSWGS84),
    crs="EPSG:4326"
).to_crs("EPSG:32632")

#Unfälle auf Berlin beschränken 
berlin_bounds = edges_gdf.total_bounds
acc_gdf_berlin = acc_gdf.cx[berlin_bounds[0]:berlin_bounds[2], berlin_bounds[1]:berlin_bounds[3]]


#Puffer um die Kanten erstellen (50 m)
edges_buffer = edges_gdf.copy()
edges_buffer.geometry = edges_buffer.geometry.buffer(50)


#Welche Unfälle liegen im Puffer der Kante
acc_joined = gpd.sjoin(acc_gdf_berlin, edges_buffer, predicate='within')
acc_counts = acc_joined.groupby(['u', 'v']).size()

#Kanten aktualisieren und normalisieren 
edges_gdf['AccRate'] = acc_counts
edges_gdf['AccRate'] = edges_gdf['AccRate'].fillna(0) 

min_acc = edges_gdf['AccRate'].min()
max_acc = edges_gdf['AccRate'].max()

if max_acc > 0:
    edges_gdf['AccRate_norm'] = (edges_gdf['AccRate'] - min_acc) / (max_acc - min_acc)
else:
    edges_gdf['AccRate_norm'] = 0.0

print("Unfälle erfolgreich gemappt und normalisiert!")
print(f"Maximale Unfallrate: {max_acc}, Minimale Unfallrate: {min_acc}")
print(f"Beispielkanten mit Unfallrate:\n{edges_gdf[['AccRate', 'AccRate_norm']].head()}")

# acc_edges_forward = set(zip(acc_df['u'], acc_df['v']))
# acc_edges_reverse = set(zip(acc_df['v'], acc_df['u']))
# acc_edges_all = acc_edges_forward.union(acc_edges_reverse)

# # 3. Schnittmenge berechnen (Wie viele Unfall-Kanten gibt es im Netz?)
# matching_edges = acc_edges_all.intersection(network_edges)

# # 4. Auswertung
# match_rate = len(matching_edges) / len(acc_edges_forward) * 100

# print(f"Kanten im Straßennetz: {len(network_edges):,}")
# print(f"Kanten in Unfall-CSV:  {len(acc_edges_forward):,}")
# print(f"Davon im Netz gefunden: {len(matching_edges):,} ({match_rate:.2f}%)")

NameError: name 'edges_gdf' is not defined

In [ ]:
#Unnfalldaten laden und als GeoDataFrame aufbauen
acc_df = read_csv("C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\data\processed\Accidents.csv", index_col=0, sep=";")
acc_gdf = gpd.GeoDataFrame(
    acc_df,
    geometry=gpd.points_from_xy(acc_df.XGCSWGS84, acc_df.YGCSWGS84),
    crs="EPSG:4326"
).to_crs("EPSG:32632")

# Puffer um die Kanten erstellen (z.B. 50 m) um Unfälle zu zählen, die in der Nähe der Kante passiert sind
edges_buffer = edges_gdf.copy()
edges_buffer.geometry = edges_buffer.geometry.buffer(50)

# Unfälle zählen, die im Puffer jeder Kante liegen 
acc_joined = gpd.sjoin(acc_gdf, edges_buffer, predicate='within')
acc_counts = acc_joined.groupby(['u', 'v']).size()

# Die Kanten bekommen jetzt die Unfallzahlen und werden normalisiert (Werte zwischen 0 und 1)
edges_gdf['AccRate'] = acc_counts
edges_gdf['AccRate'] = edges_gdf['AccRate'].fillna(0) # Kanten ohne Unfall = 0

min_acc, max_acc = edges_gdf['AccRate'].min(), edges_gdf['AccRate'].max()
edges_gdf['AccRate_norm'] = (edges_gdf['AccRate'] - min_acc) / (max_acc - min_acc)

In [ ]:
#Naturschutznähe 
nature_df = read_csv("C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\data\processed\nature_reserves_filtered.csv", index_col=0, sep=",")

#Umwandeln der String in exhte Geometrien
nature_df['geometry'] = nature_df['geometry'].apply(wkt.loads)

#Erstellen eiens GeoDataFrames und Setzen des Koordinatensystems auf UTM Zone 32N (für Distanzberechnung in Deutschland)
nature_gdf = gpd.GeoDataFrame(nature_df, geometry='geometry', crs="EPSG:4326")
nature_gdf = nature_gdf.to_crs("EPSG:32632")

# Distanzen zwischen Kanten und Naturschutzgebieten berechnen (nächster Abstand) sjoin_nearest sucht das nächste Polygon (Also Naturschutzgebiet) 
edges_with_nature = gpd.sjoin_nearest(
    edges_gdf, 
    nature_gdf, 
    how="left", 
    distance_col="dist_to_nature"
)

#Falls es Duplikate gibt, weil eine Kante genau zwischen zwei Gebieten liegt, nehmen wir nur eins 
edges_with_nature = edges_with_nature[~edges_with_nature.index.duplicated(keep='first')]

# Distanz dem Hauptdataframe hinzufügen und normalisieren (Werte zwischen 0 und 1)
edges_gdf['Dist_to_NatRes'] = edges_with_nature['dist_to_nature']
max_dist = edges_gdf['Dist_to_NatRes'].max()
min_dist = edges_gdf['Dist_to_NatRes'].min()

#Invertieren der Distanz, damit Distanz von 0m höchsten Risiko wert hat 
edges_gdf['NatRes_norm'] = 1 - ((edges_gdf['Dist_to_NatRes'] - min_dist) / (max_dist - min_dist))

In [ ]:
#Bevölkerungsdichte 
pop_df = pd.read_csv("density.csv", sep=";")

# Punkte erstellen und projizieren
pop_gdf = gpd.GeoDataFrame(
    pop_df,
    geometry=gpd.points_from_xy(pop_df['Spalte_X'], pop_df['Spalte_Y']),
    crs="EPSG:4326"  # Oder EPSG:3035, falls das JRC Raster im LAEA Format vorliegt!
).to_crs("EPSG:32632")

# --- 2. Spatial Join mit gepufferten Kanten ---
# Wir nutzen den bereits erstellten edges_buffer aus deinem Unfall-Code
pop_joined = gpd.sjoin(pop_gdf, edges_buffer, predicate='within')

# Durchschnittliche Bevölkerungsdichte für jede Kante berechnen
# (Mean ist besser als Sum, da längere Kanten sonst ungerechtfertigt bestraft werden)
pop_mean = pop_joined.groupby(['u', 'v'])['Spalte_Dichte'].mean()

# In den Haupt-DataFrame übernehmen (Kanten ohne Bevölkerung erhalten 0)
edges_gdf['PopDens'] = pop_mean
edges_gdf['PopDens'] = edges_gdf['PopDens'].fillna(0)

# --- 3. Normalisierung ---
min_pop = edges_gdf['PopDens'].min()
max_pop = edges_gdf['PopDens'].max()

# Min-Max-Skalierung
if max_pop > 0:
    edges_gdf['PopDens_norm'] = (edges_gdf['PopDens'] - min_pop) / (max_pop - min_pop)
else:
    edges_gdf['PopDens_norm'] = 0.0

In [ ]:
# Risikoscore berechnen
alpha, beta, gamma = 0.4, 0.4, 0.2
edges_gdf['Risk'] = (alpha * edges_gdf['PopDens_norm'] + 
                     beta * edges_gdf['AccRate_norm'] + 
                     gamma * edges_gdf['NatRes_norm'])

# Da die Gefahrgutklasse k im Dummy noch im Dictionary-Schlüssel stand:
Risk = {}
for e in E:
    for k in K:
        # Hier könntest du klassenspezifische Aufschläge einbauen
        Risk[(e, k)] = edges_gdf.loc[e, 'Risk']

Sets für das Modell erstellen       

In [ ]:


#Kantenlänge als Dictionary für PuLP
Dist = edges_gdf.set_index(['u', 'v'])['Dist_km'].to_dict()

print("Anzahl Kanten (E):", len(E))
print("Anzahl Knoten (N):", len(N))


#Menge der Lieferungen 
deliveries_df = pd.read_csv("deliveries.csv", index_col=0, sep=";")
D = list(deliveries_df.index)
Dem = deliveries_df['demand'].to_dict()
Class = deliveries_df['class'].to_dict()

#Start und Ziel der Lieferung identifizieren (muss ein echter Knoten sein aus N)
O_dep = deliveries_df['O'].to_dict()
D_dep = deliveries_df['D'].to_dict()


#Fahrzeugdaten 
V = ['MAN_eTGX', 'Volvo_FH_Electric', 'Mercedes_eActros_600']


#Mögliche Gefahrgutklassen 
K = ['klasse_3', 'klasse_6', 'klasse_8']

NameError: name 'edges_gdf' is not defined